Iteration 5 tests a simpler approach:

FAISS Top-5 → Cross-Encoder → reorder the same 5 chunks

The reranker is only allowed to change the order of the five FAISS results. It cannot introduce new chunks or remove any of the original Top-5 candidates.
This directly tests whether reranking can improve ordering while preserving Top-5 retrieval recall.


In [1]:
# Install dependencies
!pip -q install sentence-transformers faiss-cpu pandas numpy



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from IPython.display import display

print("Libraries loaded.")


c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.


In [15]:
# Paths
chunks_path = "./02_rmit_assessment_support_chunks.csv"
test_path = "./test_prompts.json"

# Load chunks
chunks_df = pd.read_csv(chunks_path)

# Load test prompts
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_df = pd.DataFrame(test_data["tests"])

print("Chunks:", len(chunks_df))
print("Test prompts:", len(test_df))


Chunks: 71
Test prompts: 12


## 1. Recreate the Iteration 2 embedding representation

IMPORTANT: Do not add the title here.
The exact representation is:

category + section + content


In [16]:
# EXACT Iteration 2 embedding text
chunks_df["embedding_text"] = (
    chunks_df["category"].fillna("") + ". " +
    chunks_df["section"].fillna("") + ". " +
    chunks_df["content"].fillna("")
)

# Same embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings
embeddings = embedding_model.encode(
    chunks_df["embedding_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

# Cosine similarity through normalised inner product
faiss.normalize_L2(embeddings)

# Same FAISS index type
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)

print("FAISS index created.")
print("Vectors:", faiss_index.ntotal)
print("Dimension:", embeddings.shape[1])


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.87it/s]

FAISS index created.
Vectors: 71
Dimension: 384


In [17]:
# Load cross-encoder
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Cross-encoder loaded.")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3648.52it/s]


Cross-encoder loaded.


In [23]:
def faiss_top5(question):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(
        query_embedding,
        5
    )

    results = []

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1
    ):
        idx = int(idx)
        row = chunks_df.iloc[idx]

        results.append({
            "index": idx,
            "faiss_rank": rank,
            "chunk_id": row["chunk_id"],
            "source_id": row["source_id"],
            "category": row["category"],
            "section": row["section"],
            "content": row["content"],
            "embedding_text": row["embedding_text"],
            "faiss_score": float(score)
        })

    return pd.DataFrame(results)

In [24]:
def rerank_faiss_top5(question):
    candidates = faiss_top5(question).copy()

    pairs = [
        (
            question,
            row["embedding_text"]
        )
        for _, row in candidates.iterrows()
    ]

    reranker_scores = reranker.predict(pairs)

    candidates["reranker_score"] = reranker_scores

    candidates = candidates.sort_values(
        "reranker_score",
        ascending=False
    ).reset_index(drop=True)

    candidates["reranked_position"] = (
        np.arange(len(candidates)) + 1
    )

    return candidates

## 2. Check the three previously problematic questions

This lets us verify whether the expected chunks are already in the FAISS Top-5 and whether reranking moves them up or down.


In [25]:
problem_questions = [
    (
        "Hard drive",
        "So my hard drive with my assignment stopped working and I didn't backup my work, will I be able to get an extension or special consideration?",
        ["SC-01_16"]
    ),
    (
        "Bereavement",
        "My nan just passed away but I've got a group assignment due friday, can I even apply for an extension on something like this? How much extra time could I get?",
        ["POL-01_62", "EXT-01_1"]
    ),
    (
        "ELP extension",
        "The extensions adjustment is on my ELP, but the assignment brief also says to fill out the standard extension form, do I need to do both, or is my learning plan enough on its own",
        ["EXT-01_2", "EAA-02_39"]
    )
]

for name, question, expected in problem_questions:

    print("\n" + "=" * 80)
    print(name)
    print("Expected:", expected)

    results = rerank_faiss_top5(question)

    display(
        results[
            [
                "faiss_rank",
                "reranked_position",
                "chunk_id",
                "faiss_score",
                "reranker_score"
            ]
        ]
    )



Hard drive
Expected: ['SC-01_16']


,faiss_rank,reranked_position,chunk_id,faiss_score,reranker_score
0,2,1,EXT-01_0,0.471793,-4.844401
1,1,2,EXT-01_1,0.481436,-5.289862
2,3,3,EAA-02_34,0.433386,-5.668898
3,5,4,EAA-02_38,0.415588,-6.256051
4,4,5,EAA-02_35,0.421478,-8.184461



Bereavement
Expected: ['POL-01_62', 'EXT-01_1']


,faiss_rank,reranked_position,chunk_id,faiss_score,reranker_score
0,2,1,EAA-02_41,0.521080,-0.971053
1,3,2,EAA-02_34,0.502380,-2.761659
2,1,3,EXT-01_0,0.528654,-3.953712
3,5,4,POL-01_62,0.490840,-6.734143
4,4,5,EXT-01_1,0.499601,-7.266636



ELP extension
Expected: ['EXT-01_2', 'EAA-02_39']


,faiss_rank,reranked_position,chunk_id,faiss_score,reranker_score
0,4,1,EAA-02_34,0.584328,3.315585
1,2,2,EAA-02_36,0.592427,2.216956
2,3,3,EAA-02_37,0.587779,-0.159095
3,1,4,EAA-02_39,0.602267,-0.321993
4,5,5,EAA-02_40,0.575118,-1.383297


In [26]:
def any_expected_retrieved(retrieved, expected):
    return any(chunk in retrieved for chunk in expected)


def all_expected_retrieved(retrieved, expected):
    return all(chunk in retrieved for chunk in expected)


In [27]:
def evaluate_corrected_iteration5(test_df):
    results = []

    for _, test in test_df.iterrows():

        question = test["prompt"]
        expected_chunks = test["chunks"]

        retrieved_df = rerank_faiss_top5(question)

        retrieved_chunks = retrieved_df["chunk_id"].tolist()

        top1 = retrieved_chunks[:1]
        top3 = retrieved_chunks[:3]
        top5 = retrieved_chunks[:5]

        expected_ranks = [
            retrieved_chunks.index(chunk) + 1
            for chunk in expected_chunks
            if chunk in retrieved_chunks
        ]

        results.append({
            "prompt": question,
            "expected_chunks": expected_chunks,

            "top1_chunk": top1[0] if top1 else None,
            "top3_chunks": top3,
            "top5_chunks": top5,

            "expected_rank_in_top5":
                min(expected_ranks) if expected_ranks else None,

            "top1_hit":
                any_expected_retrieved(top1, expected_chunks),
            "top3_hit":
                any_expected_retrieved(top3, expected_chunks),
            "top5_hit":
                any_expected_retrieved(top5, expected_chunks),

            "top1_all_hit":
                all_expected_retrieved(top1, expected_chunks),
            "top3_all_hit":
                all_expected_retrieved(top3, expected_chunks),
            "top5_all_hit":
                all_expected_retrieved(top5, expected_chunks)
        })

    return pd.DataFrame(results)


In [28]:
iteration5_corrected_df = evaluate_corrected_iteration5(test_df)

print("CORRECTED ITERATION 5")
print("---------------------")

print("\nAny expected chunk retrieved:")
print(f"Top-1: {iteration5_corrected_df['top1_hit'].mean():.1%}")
print(f"Top-3: {iteration5_corrected_df['top3_hit'].mean():.1%}")
print(f"Top-5: {iteration5_corrected_df['top5_hit'].mean():.1%}")

print("\nAll expected chunks retrieved:")
print(f"Top-1: {iteration5_corrected_df['top1_all_hit'].mean():.1%}")
print(f"Top-3: {iteration5_corrected_df['top3_all_hit'].mean():.1%}")
print(f"Top-5: {iteration5_corrected_df['top5_all_hit'].mean():.1%}")


CORRECTED ITERATION 5
---------------------

Any expected chunk retrieved:
Top-1: 58.3%
Top-3: 66.7%
Top-5: 91.7%

All expected chunks retrieved:
Top-1: 50.0%
Top-3: 66.7%
Top-5: 75.0%


In [29]:
comparison_df = pd.DataFrame({
    "Metric": [
        "Any expected — Top-1",
        "Any expected — Top-3",
        "Any expected — Top-5",
        "All expected — Top-1",
        "All expected — Top-3",
        "All expected — Top-5"
    ],
    "Iteration 2": [
        0.500,
        0.750,
        0.917,
        0.417,
        0.583,
        0.750
    ],
    "Corrected Iteration 5": [
        iteration5_corrected_df["top1_hit"].mean(),
        iteration5_corrected_df["top3_hit"].mean(),
        iteration5_corrected_df["top5_hit"].mean(),
        iteration5_corrected_df["top1_all_hit"].mean(),
        iteration5_corrected_df["top3_all_hit"].mean(),
        iteration5_corrected_df["top5_all_hit"].mean()
    ]
})

comparison_df["Change"] = (
    comparison_df["Corrected Iteration 5"] -
    comparison_df["Iteration 2"]
)

display(
    comparison_df.style.format({
        "Iteration 2": "{:.1%}",
        "Corrected Iteration 5": "{:.1%}",
        "Change": "{:+.1%}"
    })
)


,Metric,Iteration 2,Corrected Iteration 5,Change
0,Any expected — Top-1,50.0%,58.3%,+8.3%
1,Any expected — Top-3,75.0%,66.7%,-8.3%
2,Any expected — Top-5,91.7%,91.7%,-0.0%
3,All expected — Top-1,41.7%,50.0%,+8.3%
4,All expected — Top-3,58.3%,66.7%,+8.4%
5,All expected — Top-5,75.0%,75.0%,+0.0%


In [30]:
# Top-1 failures
top1_failures = iteration5_corrected_df[
    iteration5_corrected_df["top1_hit"] == False
][[
    "prompt",
    "expected_chunks",
    "top1_chunk",
    "top3_chunks",
    "top5_chunks",
    "expected_rank_in_top5"
]]

print("Corrected Iteration 5 Top-1 failures:", len(top1_failures))
display(top1_failures)


Corrected Iteration 5 Top-1 failures: 5


,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5
2,So my hard drive with my assignment stopped wo...,[SC-01_16],EXT-01_0,"[EXT-01_0, EXT-01_1, EAA-02_34]","[EXT-01_0, EXT-01_1, EAA-02_34, EAA-02_38, EAA...",NaN
4,I emailed my course coordinator three days ago...,[EAA-02_38],EAA-02_34,"[EAA-02_34, EAA-02_38, EXT-01_0]","[EAA-02_34, EAA-02_38, EXT-01_0, POL-01_62, EA...",2.0
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]",EAA-02_41,"[EAA-02_41, EAA-02_34, EXT-01_0]","[EAA-02_41, EAA-02_34, EXT-01_0, POL-01_62, EX...",4.0
10,I think my tutor marked me down assuming I pla...,"[POL-01_58, POL-01_59]",POL-01_70,"[POL-01_70, EAA-02_38, EXT-01_3]","[POL-01_70, EAA-02_38, EXT-01_3, POL-01_59, EX...",4.0
11,"The extensions adjustment is on my ELP, but th...","[EXT-01_2, EAA-02_39]",EAA-02_34,"[EAA-02_34, EAA-02_36, EAA-02_37]","[EAA-02_34, EAA-02_36, EAA-02_37, EAA-02_39, E...",4.0


In [31]:
# Top-5 failures
top5_failures = iteration5_corrected_df[
    iteration5_corrected_df["top5_hit"] == False
][[
    "prompt",
    "expected_chunks",
    "top1_chunk",
    "top3_chunks",
    "top5_chunks",
    "expected_rank_in_top5"
]]

print("Corrected Iteration 5 Top-5 failures:", len(top5_failures))
display(top5_failures)


Corrected Iteration 5 Top-5 failures: 1


,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5
2,So my hard drive with my assignment stopped wo...,[SC-01_16],EXT-01_0,"[EXT-01_0, EXT-01_1, EAA-02_34]","[EXT-01_0, EXT-01_1, EAA-02_34, EAA-02_38, EAA...",NaN
